# Question 5 - LLM Judge Test

```text
User question
      ↓
SQL Agent
      ↓
get_schema()
      ↓
run_sql()
      ↓
SQLResult
      ↓
Judge LLM
      ↓
natural-language criteria
      ↓
PASS / FAIL
```

- Question 3 → deterministic result assertion
- Question 4 → deterministic tool-behavior assertion
- Question 5 → semantic LLM-judge assertion

The separate judge receives the structured agent output, ordered application tool calls, and actual tool results. It evaluates the three natural-language criteria without SQL tools of its own. A judge can make mistakes, so we also check the answer independently in DuckDB.

Run from `week3_homework/` with the project dependencies and `OPENAI_API_KEY` available in the environment.

In [1]:
from sql_agent import agent
from utils import collect_tools
from judge import evaluate_agent_performance

criteria = [
    "the SQL query correctly calculates average fare by hour of day",
    "the result identifies a specific hour as having the highest average fare",
    "the result includes the actual average fare amount",
]
result = await agent.run("Which hour of the day has the highest average fare amount?")
print("Agent-generated SQL:")
print(result.output.sql_query)
print("result_text:")
print(result.output.result_text)
print("row_count:", result.output.row_count)
print("Structured SQLResult:")
print(result.output.model_dump_json(indent=2))

                 pydantic-ai v2.63.0 • Python 3.13.15

      / \        agent: agent • model: openai:gpt-4o-mini • output: SQLResult • tools: 2
     /   \         capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Pydantic Logfire: uvx logfire-cli wizard --task pydantic-ai
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


Agent-generated SQL:
SELECT EXTRACT(HOUR FROM tpep_pickup_datetime) AS pickup_hour, AVG(fare_amount) AS average_fare
FROM trips
GROUP BY pickup_hour
ORDER BY average_fare DESC
LIMIT 1;
result_text:
pickup_hour,average_fare
5,26.619918460882545
row_count: 1
Structured SQLResult:
{
  "sql_query": "SELECT EXTRACT(HOUR FROM tpep_pickup_datetime) AS pickup_hour, AVG(fare_amount) AS average_fare\nFROM trips\nGROUP BY pickup_hour\nORDER BY average_fare DESC\nLIMIT 1;",
  "result_text": "pickup_hour,average_fare\n5,26.619918460882545",
  "row_count": 1
}


In [2]:
tool_calls = collect_tools(result.new_messages())
for index, call in enumerate(tool_calls, start=1):
    print(f"{index}. {call.name}({call.args})")
assert tool_calls[0].name == "get_schema"
assert any(call.name == "run_sql" for call in tool_calls)

1. get_schema({})
2. run_sql({'query': 'SELECT EXTRACT(HOUR FROM tpep_pickup_datetime) AS pickup_hour, AVG(fare_amount) AS average_fare\nFROM trips\nGROUP BY pickup_hour\nORDER BY average_fare DESC\nLIMIT 1;'})


In [3]:
feedback = await evaluate_agent_performance(result, criteria)
print("Overall judge feedback:")
print(feedback.feedback)
assert len(feedback.criteria) == len(criteria)
for criterion in feedback.criteria:
    print(f"{'PASS' if criterion.passed else 'FAIL'}: {criterion.criterion_description}")
    print(criterion.judgement)
    assert criterion.passed, f"{criterion.criterion_description}: {criterion.judgement}"

Overall judge feedback:
The SQL agent successfully produced a correct SQL query and interpretation of the results based on the provided criteria, accurately calculating the average fare by hour and identifying the hour with the highest fare.
PASS: the SQL query correctly calculates average fare by hour of day
The SQL query correctly uses the EXTRACT function to get the hour from the pickup datetime and calculates the average fare using AVG, grouped by hour.
PASS: the result identifies a specific hour as having the highest average fare
The output includes a specific hour (5) that has the highest average fare, as the query orders results by average fare in descending order and limits to 1 result.
PASS: the result includes the actual average fare amount
The result shows the average fare amount of 26.619918460882545 for hour 5.


This independent DuckDB query provides a separate check of the actual answer. The pytest test uses the reusable `assert_criteria` function in `judge.py`, which calls the same judge evaluation and prints its feedback.

In [4]:
from sql_tools import con

verification_query = """SELECT
    EXTRACT(HOUR FROM tpep_pickup_datetime) AS hour,
    AVG(fare_amount) AS average_fare
FROM trips
GROUP BY hour
ORDER BY average_fare DESC
LIMIT 1;"""
winning_hour, average_fare = con.execute(verification_query).fetchone()
print(verification_query)
print("Winning hour:", winning_hour)
print("Average fare amount:", average_fare)

SELECT
    EXTRACT(HOUR FROM tpep_pickup_datetime) AS hour,
    AVG(fare_amount) AS average_fare
FROM trips
GROUP BY hour
ORDER BY average_fare DESC
LIMIT 1;
Winning hour: 5
Average fare amount: 26.61991846088254


**Hour with the highest average fare amount: 5 (05:00–05:59).**

**Average fare amount: `26.61991846088254`.**

These values come from the independent DuckDB query above; the live SQL agent also identified hour 5 and the same average fare. All three judge criteria passed. Floating-point aggregation can differ in the last few digits between runs.